# Clasificador de noticias

En esta práctica crearemos un clasificador de noticias con técnicas de procesamiento del lenguaje natural vistas en clase, centrándonos en la representación del texto.

Utilizaremos el conjunto de datos [AG News](https://www.kaggle.com/amananandrai/ag-news-classification-dataset), que contiene un millón de noticias de cuatro categorías.

## Conjunto de datos

Cargaremos el conjunto con la biblioteca `tensorflow_datasets`, que también facilita la separación en datos de entrenamiento y de prueba.

La función `load('ag_news_subset')` de `tensorflow_datasets` devuelve un `tf.data.Dataset` con los ejemplos. Este objeto se parece a un `numpy.array`, pero también contiene las etiquetas. Accederemos a los ejemplos y las etiquetas mediante los atributos `data` y `label`.

## Preparación del conjunto de datos

Para instalar `tensorflow_datasets`, ejecutaremos la siguiente celda. Esta biblioteca incluye muchos conjuntos de datos útiles para la asignatura.

In [22]:
# Instalem les llibreries necessàries

!pip install keras
!pip install tensorflow
!pip install tensorflow_datasets

  Using cached keras-3.8.0-py3-none-any.whl.metadata (5.8 kB)
Using cached keras-3.8.0-py3-none-any.whl (1.3 MB)
  Attempting uninstall: keras
    Found existing installation: keras 2.15.0
    Uninstalling keras-2.15.0:
      Successfully uninstalled keras-2.15.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.15.1 requires keras<2.16,>=2.15.0, but you have keras 3.8.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: pip install --upgrade pip
  Using cached keras-2.15.0-py3-none-any.whl.metadata (2.4 kB)
Using cached keras-2.15.0-py3-none-any.whl (1.7 MB)
  Attempting uninstall: keras
    Found existing installation: keras 3.8.0
    Uninstalling keras-3.8.0:
      Successfully uninstalled keras-3.8.0

[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: pip instal

In [23]:
import keras
import tensorflow as tf
import tensorflow_datasets as tfds

# En aquest tutorial, entrenarem molts models. Per utilitzar la memòria de la GPU de forma eficient,
# configurarem tensorflow perquè creixi la memòria de la GPU quan sigui necessari.
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

# Carreguem el dataset. Es descarregarà automàticament i es guardarà a la carpeta
# /home/USUARI/tensorflow_datasets/ag_news_subset/1.0.0
dataset = tfds.load('ag_news_subset')

Automáticamente, la función `load` ha dividido los datos en dos conjuntos: entrenamiento (`train`) y prueba (`test`). Son objetos `tf.data.Dataset` que contienen los ejemplos y las etiquetas. Accederemos a ellos mediante los atributos `data` y `label`.

In [24]:
# Separem el dataset en conjunt d'entrenament i de test
ds_train = dataset['train']
ds_test = dataset['test']
# Vejam quants exemples hi ha en cada conjunt
print('Nombre d\'exemples de train:', len(ds_train))
print('Nombre d\'exemples de test:', len(ds_test))

Nombre d'exemples de train: 120000
Nombre d'exemples de test: 7600


Imprimimos los primeros cinco ejemplos del conjunto de entrenamiento. Cada ejemplo contiene una noticia y su etiqueta.

In [25]:
classes = ['World', 'Sports', 'Business', 'Sci/Tech']

# Imprimim els primers 5 exemples del conjunt d'entrenament
for w in ds_train.take(5):
    print(f"{w['label']} ({classes[w['label']]}) -> {w['title']} {w['description']}")

3 (Sci/Tech) -> b'AMD Debuts Dual-Core Opteron Processor' b'AMD #39;s new dual-core Opteron chip is designed mainly for corporate computing applications, including databases, Web services, and financial transactions.'
1 (Sports) -> b"Wood's Suspension Upheld (Reuters)" b'Reuters - Major League Baseball\\Monday announced a decision on the appeal filed by Chicago Cubs\\pitcher Kerry Wood regarding a suspension stemming from an\\incident earlier this season.'
2 (Business) -> b'Bush reform may have blue states seeing red' b'President Bush #39;s  quot;revenue-neutral quot; tax reform needs losers to balance its winners, and people claiming the federal deduction for state and local taxes may be in administration planners #39; sights, news reports say.'
3 (Sci/Tech) -> b"'Halt science decline in schools'" b'Britain will run out of leading scientists unless science education is improved, says Professor Colin Pillinger.'
1 (Sports) -> b'Gerrard leaves practice' b'London, England (Sports Network

2025-02-03 21:30:59.025943: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


## Representación del texto

Para entrenar una red neuronal, debemos representar el texto con vectores numéricos. En esta práctica utilizaremos Bag of Words (BoW), que representa cada palabra mediante un número. Es una representación sencilla: no tiene en cuenta el orden de las palabras ni su semántica, pero funciona razonablemente bien en muchos casos.

### Limitar el vocabulario

El conjunto contiene muchas palabras poco frecuentes. No suelen ser útiles para entrenar el modelo y aumentan la dimensionalidad de la representación. Por eso limitaremos el vocabulario a las 50 000 palabras más frecuentes.

Utilizaremos la capa `TextVectorization` de Keras para convertir el texto en secuencias numéricas, limitar el vocabulario y pasar las palabras a minúsculas mediante los parámetros `max_tokens` y `standardize`.

> Para crear el vocabulario utilizaremos solo 500 noticias del conjunto de entrenamiento, porque este proceso es lento y queremos reducir el tiempo de ejecución del tutorial. En un caso real usaríamos todas las noticias de entrenamiento. Existe el riesgo de que alguna palabra del conjunto de prueba no aparezca en el vocabulario y reduzca la exactitud del modelo, pero no debería ser un problema grave.

In [26]:
# Limitem el vocabulari a les 50.000 paraules més freqüents
tamany_vocabulari = 50000

# Creem la capa TextVectorization
# max_tokens: nombre màxim de paraules que tindrà el vocabulari
# standardize: funció que s'aplicarà a cada paraula per convertir-la a minúscules
# output_mode: 'int' perquè la capa retorni seqüències de nombres (per defecte)
vectorize_layer = keras.layers.TextVectorization(
    max_tokens=tamany_vocabulari,
    standardize='lower_and_strip_punctuation',
    output_mode='int'
)

# Actualitzem el vocabulari de la capa TextVectorization amb les paraules del conjunt d'entrenament
# Per fer-ho, primer creem un dataset amb les notícies del conjunt d'entrenament. Per reduïr el temps d'execució solament treballarem en 500 notícies
ds_text = ds_train.take(500).map(lambda article: article['title'] + ' ' + article['description'])

# Després, apliquem la capa TextVectorization a aquest dataset per construir el vocabulari
vectorize_layer.adapt(ds_text)

2025-02-03 21:30:59.364616: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


Ahora podemos acceder al vocabulario de la capa `TextVectorization` mediante el atributo `get_vocabulary`. Devuelve una lista de las palabras. Como vemos, la capa ha pasado las palabras a minúsculas y ha eliminado los signos de puntuación.

In [27]:
# Imprimim les 10 primeres paraules del vocabulari
print(vectorize_layer.get_vocabulary()[:10])

# Imprimim el tamany del vocabulari
tamany_vocabulari = len(vectorize_layer.get_vocabulary())
print('Tamany del vocabulari:', tamany_vocabulari)

['', '[UNK]', 'the', 'to', 'a', 'in', 'of', 'and', 'on', 'for']
Tamany del vocabulari: 5335


El vectorizador convierte un texto en una secuencia de números. Veamos un ejemplo.

In [28]:
vectorize_layer('this is a test sentence')

<tf.Tensor: shape=(5,), dtype=int64, numpy=array([ 35,  17,   4, 675,   1])>

## Representación Bag of Words

Aunque el significado de las palabras depende del contexto, BoW puede ser útil en algunos casos. Por ejemplo, en una noticia, la palabra `covid` puede indicar que trata sobre la pandemia y la palabra `snow` puede señalar que trata sobre el tiempo atmosférico.

BoW es una de las técnicas clásicas de vectorización más sencillas. Primero creamos un vector de ceros, con una posición por cada palabra del vocabulario. Después, por cada palabra del texto, incrementamos en uno la posición correspondiente. Por ejemplo, el texto `this sentence is a test sentence` podría representarse como `[1, 2, 1, 1, 0, 0, 0, 0, 0, 0, ...]`.

BoW se parece a la representación *one-hot*. La diferencia es que *one-hot* genera un vector por palabra con un solo 1, mientras que BoW cuenta cuántas veces aparece cada palabra. Por tanto, el vector BoW puede entenderse como la suma de vectores *one-hot*.

Por ejemplo, si el vector *one-hot* de la primera palabra es `[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...]` y el de la segunda es `[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...]`, su suma es `[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...]`.

Para generar la representación BoW, convertiremos cada palabra en un vector *one-hot* y sumaremos los vectores. Lo haremos con la función `to_bow`, que recibe un texto y devuelve su vector BoW.

In [29]:
def to_bow(text):
    return tf.reduce_sum(tf.one_hot(vectorize_layer(text), tamany_vocabulari), axis=0)


to_bow('My dog likes hot dogs on a hot day.').numpy()

array([0., 5., 0., ..., 0., 0., 0.], dtype=float32)

### Entrenamiento del clasificador BoW

Utilizaremos `TextVectorization` para convertir el texto en una representación BoW y una capa `Dense` para clasificarlo. Esta capa totalmente conectada tendrá tantas neuronas de entrada como palabras haya en el vocabulario y cuatro neuronas de salida, una por cada categoría.

### Representación BoW

Convertimos cada texto a BoW con la función `to_bow` creada anteriormente.

Aplicaremos esta función a cada ejemplo mediante `map` de `tf.data.Dataset`. Después agruparemos los ejemplos en lotes de 128 con `batch` para hacer más eficiente el entrenamiento.

In [30]:
batch_size = 128

ds_train_bow = ds_train.map(lambda x: (to_bow(x['title'] + ' ' + x['description']), x['label'])).batch(batch_size)
ds_test_bow = ds_test.map(lambda x: (to_bow(x['title'] + ' ' + x['description']), x['label'])).batch(batch_size)

### Modelo de clasificación

Crearemos el modelo con la clase `Sequential` de Keras, que permite definir una secuencia de capas.

El modelo tendrá una capa `Dense` con cuatro neuronas de salida y tantas neuronas de entrada como palabras haya en el vocabulario (`input_shape=(tamany_vocabulari,)`). La función `softmax` devuelve una distribución de probabilidades entre las cuatro categorías.

Compilaremos el modelo con el optimizador `Adam` y la función de pérdida `SparseCategoricalCrossentropy`, adecuada para una clasificación multiclase. También indicaremos que queremos calcular la exactitud.

Por último, entrenaremos el modelo con `fit`, utilizando los conjuntos de entrenamiento y validación. La función calculará la exactitud de validación en cada época para que podamos observar su evolución.

In [31]:
model = keras.Sequential([
    keras.layers.Dense(4, input_shape=(tamany_vocabulari,), activation='softmax')
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.fit(ds_train_bow, validation_data=ds_test_bow)

938/938 [==============================] - 18s 19ms/step - loss: 0.5912 - accuracy: 0.8502 - val_loss: 0.4162 - val_accuracy: 0.8774


El modelo alcanza una exactitud superior a 0,86 en el conjunto de entrenamiento. Es un resultado razonable, teniendo en cuenta que hemos simplificado el problema para reducir el tiempo de ejecución del tutorial. En un caso real usaríamos todas las noticias de entrenamiento y podríamos obtener un modelo más preciso.

### Integrar el clasificador en una red neuronal

Como hemos definido el vectorizador como una capa de Keras, podemos crear un único modelo que incluya tanto el vectorizador como el clasificador. Así podremos entrenarlos conjuntamente y guardar o cargar todo el modelo en un solo archivo.

El modelo tendrá estas capas:

* La capa de entrada recibe el texto de la noticia.
* `TextVectorization` convierte el texto en una secuencia de enteros.
* `one_hot` convierte los enteros en vectores *one-hot*.
* `reduce_sum` suma esos vectores para obtener la representación BoW.
* `Dense` clasifica el texto en una de las cuatro categorías.

In [32]:
def extrau_text(x):
    # Concatenem el títol i la descripció de la notícia
    return x['title'] + x['description']


def dict_a_tupla(x):
    # Convertim el diccionari en una tupla amb el text i l'etiqueta
    return (extrau_text(x), x['label'])


# Creem el model
# La capa d'entrada serà un string amb el text de la notícia
inp = keras.Input(shape=(1,), dtype=tf.string)

# Vectorize_layer retorna un tensor amb un enter per cada paraula del text
enters = vectorize_layer(inp)

# Convertim el tensor d'enters en un tensor d'arrays one-hot
one_hots = tf.one_hot(enters, tamany_vocabulari)

# Sumem els vectors one-hot per obtenir la representació BoW
bow = tf.reduce_sum(one_hots, axis=1)

# Creem la capa de sortida amb 4 neurones i funció d'activació softmax
out = keras.layers.Dense(4, activation='softmax')(bow)

# Creem el model
model = keras.models.Model(inp, out)
model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_3 (Text  (None, None)              0         
 Vectorization)                                                  
                                                                 
 tf.one_hot_1 (TFOpLambda)   (None, None, 5335)        0         
                                                                 
 tf.math.reduce_sum_1 (TFOp  (None, 5335)              0         
 Lambda)                                                         
                                                                 
 dense_6 (Dense)             (None, 4)                 21344     
                                                                 
Total params: 21344 (83.38 KB)
Trainable params: 21344 (83.

### Entrenamiento del modelo

Aplicaremos la función `dict_a_tupla` a cada ejemplo con `map` de `tf.data.Dataset`. Después agruparemos los ejemplos en lotes de 128 con `batch` para hacer más eficiente el entrenamiento.

In [33]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['acc'])
model.fit(ds_train.map(dict_a_tupla).batch(batch_size), validation_data=ds_test.map(dict_a_tupla).batch(batch_size))

938/938 [==============================] - 19s 20ms/step - loss: 0.6325 - acc: 0.8314 - val_loss: 0.4442 - val_acc: 0.8671


### Generar automáticamente los vectores BoW

En el modelo anterior construimos manualmente la representación BoW para entender cómo funciona y cómo implementarla en Keras. Sin embargo, podemos generarla directamente con el parámetro `output_mode="count"` de `TextVectorization`.

In [34]:
# Creem la xarxa amb la capa TextVectorization
model = keras.models.Sequential([
    keras.layers.experimental.preprocessing.TextVectorization(max_tokens=tamany_vocabulari, output_mode='count'),
    keras.layers.Dense(4, input_shape=(tamany_vocabulari,), activation='softmax')
])

# Actualitzem el vocabulari de la capa TextVectorization amb les paraules del conjunt d'entrenament
model.layers[0].adapt(ds_train.take(500).map(extrau_text))
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['acc'])
model.fit(ds_train.map(dict_a_tupla).batch(batch_size), validation_data=ds_test.map(dict_a_tupla).batch(batch_size))

model.summary()

938/938 [==============================] - 3s 3ms/step - loss: 0.6143 - acc: 0.8423 - val_loss: 0.4370 - val_acc: 0.8717
Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 text_vectorization_4 (Text  (None, 5335)              0         
 Vectorization)                                                  
                                                                 
 dense_7 (Dense)             (None, 4)                 21344     
                                                                 
Total params: 21344 (83.38 KB)
Trainable params: 21344 (83.38 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


### N-gramas

BoW no tiene en cuenta el orden ni la semántica de las palabras. En algunos casos podemos mejorar esta representación con n-gramas, es decir, secuencias de n palabras consecutivas. Por ejemplo, en `this sentence is a test sentence`, los bigramas son `this sentence`, `sentence is`, `is a`, `a test` y `test sentence`.

Para crear una representación BoW con n-gramas utilizaremos `CountVectorizer`. Esta función recibe una lista de textos y devuelve una matriz: cada fila corresponde a un texto y cada columna a una palabra del vocabulario. Cada celda indica cuántas veces aparece esa palabra en el texto.

In [35]:
from sklearn.feature_extraction.text import CountVectorizer

bigram_vectorizer = CountVectorizer(ngram_range=(1, 2), token_pattern=r'\b\w+\b', min_df=1)
corpus = [
    'I like hot dogs.',
    'The dog ran fast.',
    'Its hot outside.',
]
bigram_vectorizer.fit_transform(corpus)
print("Vocabulary:\n", bigram_vectorizer.vocabulary_)
bigram_vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()

Vocabulary:
 {'i': 7, 'like': 11, 'hot': 4, 'dogs': 2, 'i like': 8, 'like hot': 12, 'hot dogs': 5, 'the': 16, 'dog': 0, 'ran': 14, 'fast': 3, 'the dog': 17, 'dog ran': 1, 'ran fast': 15, 'its': 9, 'outside': 13, 'its hot': 10, 'hot outside': 6}


array([[1, 0, 1, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

El principal problema de los n-gramas es que su número crece muy rápido con n. Por ejemplo, un vocabulario de 50 000 palabras puede generar 2 500 millones de bigramas. Por eso, en la práctica se combinan con otras técnicas de reducción de dimensionalidad, como `Word2Vec` o `GloVe`.

## Representación TF-IDF

TF-IDF es otra representación habitual en procesamiento del lenguaje natural. Tiene en cuenta la frecuencia de las palabras en el texto y en el conjunto de documentos. Así concede más peso a las palabras que caracterizan un texto.

Una palabra que aparece muchas veces en un texto, pero también en muchos otros, aporta poca información (por ejemplo, `and`). En cambio, si aparece muchas veces en un texto y en pocos otros (`covid`), puede ser más informativa.

Utilizaremos TF-IDF mediante el parámetro `output_mode="tf-idf"` de `TextVectorization`. Prueba esta opción para comprobar si mejora un poco la exactitud.

In [36]:
# Creem la xarxa amb la capa TextVectorization
model = keras.models.Sequential([
    keras.layers.experimental.preprocessing.TextVectorization(max_tokens=tamany_vocabulari, output_mode='tf-idf'),
    keras.layers.Dense(4, input_shape=(tamany_vocabulari,), activation='softmax')
])

# Actualitzem el vocabulari de la capa TextVectorization amb les paraules del conjunt d'entrenament
model.layers[0].adapt(ds_train.take(500).map(extrau_text))
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['acc'])
model.fit(ds_train.map(dict_a_tupla).batch(batch_size), validation_data=ds_test.map(dict_a_tupla).batch(batch_size))

model.summary()

938/938 [==============================] - 3s 3ms/step - loss: 0.4297 - acc: 0.8637 - val_loss: 0.3547 - val_acc: 0.8828
Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 text_vectorization_5 (Text  (None, 5335)              1         
 Vectorization)                                                  
                                                                 
 dense_8 (Dense)             (None, 4)                 21344     
                                                                 
Total params: 21345 (83.38 KB)
Trainable params: 21344 (83.38 KB)
Non-trainable params: 1 (8.00 Byte)
_________________________________________________________________


## Representación Word2Vec

Word2Vec es una representación habitual en procesamiento del lenguaje natural. Tiene en cuenta el contexto de las palabras y permite operar con sus vectores. Por ejemplo, al restar el vector de `man` al de `king` y sumar el de `woman`, se obtiene un vector cercano al de `queen`.

Utilizaremos la biblioteca `gensim` y el modelo `word2vec-google-news-300`, que contiene vectores para tres millones de palabras y frases.

> La primera vez que se ejecute esta celda, `load` descargará un modelo de 1,5 GB. Puede tardar unos minutos. Después se guardará en `/home/USUARIO/gensim-data/word2vec-google-news-300/word2vec-google-news-300.gz` y no será necesario descargarlo de nuevo.
> La función devuelve un objeto `KeyedVectors` con la representación Word2Vec.

Ahora podemos consultar la representación Word2Vec de cualquier palabra. Por ejemplo, para obtener el vector de `king`, utilizaremos `get_vector` del objeto `KeyedVectors`.

In [38]:
w2v.get_vector('king')

array([ 1.25976562e-01,  2.97851562e-02,  8.60595703e-03,  1.39648438e-01,
       -2.56347656e-02, -3.61328125e-02,  1.11816406e-01, -1.98242188e-01,
        5.12695312e-02,  3.63281250e-01, -2.42187500e-01, -3.02734375e-01,
       -1.77734375e-01, -2.49023438e-02, -1.67968750e-01, -1.69921875e-01,
        3.46679688e-02,  5.21850586e-03,  4.63867188e-02,  1.28906250e-01,
        1.36718750e-01,  1.12792969e-01,  5.95703125e-02,  1.36718750e-01,
        1.01074219e-01, -1.76757812e-01, -2.51953125e-01,  5.98144531e-02,
        3.41796875e-01, -3.11279297e-02,  1.04492188e-01,  6.17675781e-02,
        1.24511719e-01,  4.00390625e-01, -3.22265625e-01,  8.39843750e-02,
        3.90625000e-02,  5.85937500e-03,  7.03125000e-02,  1.72851562e-01,
        1.38671875e-01, -2.31445312e-01,  2.83203125e-01,  1.42578125e-01,
        3.41796875e-01, -2.39257812e-02, -1.09863281e-01,  3.32031250e-02,
       -5.46875000e-02,  1.53198242e-02, -1.62109375e-01,  1.58203125e-01,
       -2.59765625e-01,  

También podemos consultar las palabras más parecidas a una palabra. Por ejemplo, `most_similar` del objeto `KeyedVectors` devuelve las palabras más similares a `king`.

In [39]:
for w, p in w2v.most_similar('king'):
    print(f"{w} -> {p}")

kings -> 0.7138045430183411
queen -> 0.6510956883430481
monarch -> 0.6413194537162781
crown_prince -> 0.6204220056533813
prince -> 0.6159993410110474
sultan -> 0.5864824056625366
ruler -> 0.5797567367553711
princes -> 0.5646552443504333
Prince_Paras -> 0.5432944297790527
throne -> 0.5422105193138123


Lo más interesante de Word2Vec es que los vectores tienen una estructura matemática que permite realizar operaciones con palabras. Por ejemplo, al restar el vector de `man` al de `king` y sumar el de `woman`, se obtiene un vector parecido al de `queen`.

$$ KING - MAN + WOMAN = QUEEN $$

Para realizar esta operación utilizaremos `most_similar` de `KeyedVectors`, pasando los vectores de `king`, `woman` y `man`. La función devolverá una lista de las palabras más similares al vector resultante. Como vemos, `queen` aparece en primer lugar.

In [40]:
w2v.most_similar(positive=['king', 'woman'], negative=['man'])[0]

('queen', 0.7118193507194519)

In [41]:
from keras.src.layers import Embedding


def gensim_to_keras_embedding(model, train_embeddings=False):
    """Get a Keras 'Embedding' layer with weights set from Word2Vec model's learned word embeddings.

    Parameters
    ----------
    train_embeddings : bool
        If False, the returned weights are frozen and stopped from being updated.
        If True, the weights can / will be further updated in Keras.

    Returns
    -------
    `keras.layers.Embedding`
        Embedding layer, to be used as input to deeper network layers.

    """
    keyed_vectors = model  # structure holding the result of training
    weights = keyed_vectors.vectors  # vectors themselves, a 2D numpy array    
    index_to_key = keyed_vectors.index_to_key  # which row in `weights` corresponds to which word?

    layer = Embedding(
        input_dim=weights.shape[0],
        output_dim=weights.shape[1],
        weights=[weights],
        trainable=train_embeddings,
    )
    return layer

In [42]:
model = keras.models.Sequential([
    vectorize_layer,
    gensim_to_keras_embedding(w2v, train_embeddings=False),
    keras.layers.Lambda(lambda x: tf.reduce_mean(x,axis=1)),
    keras.layers.Dense(4, activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'])
model.fit(ds_train.map(dict_a_tupla).batch(128),validation_data=ds_test.map(dict_a_tupla).batch(128),epochs=5)

Epoch 1/5
938/938 [==============================] - 19s 7ms/step - loss: 1.3542 - acc: 0.4296 - val_loss: 1.3274 - val_acc: 0.4925
Epoch 2/5
938/938 [==============================] - 6s 6ms/step - loss: 1.3023 - acc: 0.4952 - val_loss: 1.2841 - val_acc: 0.5149
Epoch 3/5
938/938 [==============================] - 6s 6ms/step - loss: 1.2629 - acc: 0.5154 - val_loss: 1.2497 - val_acc: 0.5322
Epoch 4/5
938/938 [==============================] - 6s 6ms/step - loss: 1.2308 - acc: 0.5313 - val_loss: 1.2209 - val_acc: 0.5459
Epoch 5/5
938/938 [==============================] - 8s 8ms/step - loss: 1.2037 - acc: 0.5444 - val_loss: 1.1963 - val_acc: 0.5539


El resultado no es muy bueno porque el modelo Word2Vec utilizado no contiene algunas palabras del conjunto de datos. Por ejemplo, `covid` no aparece en el modelo.

Para resolverlo necesitaríamos entrenar Word2Vec con las palabras del conjunto, pero este proceso es lento y queda fuera de este tutorial.